# Домашнее задание 6
## Манёвры уклонения от столкновения.

**Введение в маневрирование КА · К лекции 9**  
**Срок сдачи: 23 ноября 2026 года, 23:59.**

Сравните опасность двух сближений с учётом неопределённости положения,
затем выберите время и величину манёвра уклонения.

**Оба задания обязательны.**


### Как работать с ноутбуком

Выполняйте ячейки по порядку. Нужны Python 3, NumPy и Matplotlib.
Готовые функции выводят таблицы, строят графики и позволяют при желании проверить ответы;
их код менять не требуется. Самостоятельно получите формулы, заполните
переменные со значением `None` и текстовые ячейки с ответами.

**Чекеры предназначены только для самопроверки. Использовать их не обязательно.**
Ячейки с их вызовами можно пропустить; сохранять результаты проверок не нужно.

**Единицы и порядок компонент:**
- длины — в **метрах**, время — в **секундах**, скорости и импульсы — в **м/с**;
- в задании 1 строки массивов соответствуют случаям **A, B**;
- `P_1` имеет форму `(2, 3)`; первая строка соответствует оси $X$, вторая — оси $Y$;
- строки `b_1` — два вектора промаха, столбцы — их компоненты в указанных осях;
- в задании 2 строки массивов соответствуют моментам $t_{m,1},t_{m,2}$;
- в `roots_2` отрицательный корень записывается первым, положительный — вторым;
- `dv_2` — три компоненты $(\Delta V_r,\Delta V_t,\Delta V_n)$ в ОСК управляемого КА;
- `selected_case_2` равен **1** или **2**, а не индексу Python;
- заданные коэффициенты влияния уже учитывают перенос импульса к сближению:
  **дополнительно умножать их на время упреждения не нужно**.

Не округляйте промежуточные результаты. Численные проверки не заменяют
выводы и объяснения. Пока ответы не заполнены, ячейки выводят напоминания
и не прерывают выполнение ноутбука.


## Общие исходные данные

Рассмотрите два случая номинального сближения без манёвра. В опорную эпоху
$t=0$ относительные положения опасного объекта относительно управляемого КА
в инерциальных осях $(X,Y,Z)$ равны:
$$\boldsymbol\rho_A(0)=(70,0,1200)^\top\ \text{м},\qquad
  \boldsymbol\rho_B(0)=(0,110,1800)^\top\ \text{м}.$$
Относительная скорость одинакова:
$$\boldsymbol v_{\rm rel}=(0,0,-20)^\top\ \text{м/с},\qquad
  \boldsymbol\rho(t)=\boldsymbol\rho(0)+\boldsymbol v_{\rm rel}t.$$

Ошибки положения двух объектов независимы, гауссовы, имеют нулевое
математическое ожидание. Их ковариации постоянны в неподвижных осях:
$$C_{\rm sc}=\operatorname{diag}(30^2,60^2,90^2)\ \text{м}^2,\qquad
  C_{\rm obj}=\operatorname{diag}(40^2,80^2,120^2)\ \text{м}^2.$$

Суммарный радиус столкновения $R_c=1$ м, допустимая вероятность
$P_{\rm доп}=5\cdot10^{-5}$. В плоскости сближения используйте оси
$X,Y$ в указанном порядке. Вероятность оценивайте приближённо:
$$P_c(\boldsymbol b)=\frac{R_c^2}{2\sqrt{\det C_b}}
\exp\!\left(-\frac12\boldsymbol b^\top C_b^{-1}\boldsymbol b\right).$$
Это приближение считает плотность почти постоянной внутри круга столкновения.
Во всём задании используется именно эта формула.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

ACCENT = "#187AA5"
SECOND = "#A36540"
INK = "#302B28"
fonts = {font.name for font in font_manager.fontManager.ttflist}
plt.rcParams.update({
    "font.family": "PT Serif" if "PT Serif" in fonts else "DejaVu Serif",
    "font.size": 11,
    "text.color": INK,
    "axes.labelcolor": INK,
    "axes.titleweight": "normal",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 110,
})


In [ ]:
RHO0 = np.array([[70.0, 0.0, 1200.0],
                 [0.0, 110.0, 1800.0]])  # строки A, B; м
V_REL = np.array([0.0, 0.0, -20.0])      # м/с
C_SC = np.diag(np.array([30.0, 60.0, 90.0]) ** 2)    # м²
C_OBJ = np.diag(np.array([40.0, 80.0, 120.0]) ** 2)   # м²
R_C = 1.0             # м
P_DOP = 5e-5

LEAD_TIMES_2 = np.array([600.0, 1800.0])  # с
G_T_2 = np.array([[1200.0, 600.0],
                  [3600.0, -1200.0]])   # с; строки — моменты манёвра
DV_MAX = 0.006         # м/с


### Готовые функции для таблиц, графиков и самопроверки

Выполните следующие ячейки. Разбирать их код для выполнения задания не нужно.

`pc_approx(b, C_b)` вычисляет вероятность по заданной формуле;
последний индекс `b` — две компоненты вектора промаха.
`encounter_table(...)` выводит результаты задания 1.
`plot_risk_boundary(b, C_b)` строит границу $P_c=P_{\rm доп}$ и отмечает
два найденных вектора промаха; масштабы осей одинаковы.

`maneuver_table(...)` выводит два корня, ближайший к нулю корень
и результат проверки ограничения для каждого момента.
`plot_maneuver_probabilities(b0, C_b, selected_case, dv)` строит две зависимости
$P_c(\Delta V_t)$ на $[-0{,}04;0{,}04]$ м/с, порог вероятности и область
допустимых импульсов. Если момент и импульс уже выбраны, отмечается их точка.

`check_task1(...)` и `check_task2(...)` возвращают `True` при успешной
численной проверке и `False`, если обнаружена ошибка.
Если ответы ещё не заполнены, возвращается `None`.


In [ ]:
def _answers_ready(*pairs):
    """Проверка формы и конечности числовых ответов перед выводом."""
    for value, shape in pairs:
        if value is None:
            return False
        try:
            array = np.asarray(value)
        except (TypeError, ValueError):
            return False
        if (array.shape != shape or array.dtype.kind not in "biuf"
                or not np.all(np.isfinite(array))):
            return False
    return True


def _covariance_ready(C_b):
    if not _answers_ready((C_b, (2, 2))):
        return False
    matrix = np.asarray(C_b, dtype=float)
    return (np.allclose(matrix, matrix.T, rtol=0, atol=1e-10)
            and bool(np.all(np.linalg.eigvalsh(matrix) > 0)))


def pc_approx(b, C_b):
    """Вероятность по заданному приближению; последний индекс b — X,Y."""
    b = np.asarray(b, dtype=float)
    C_b = np.asarray(C_b, dtype=float)
    exponent = np.einsum("...i,ij,...j->...", b, np.linalg.inv(C_b), b)
    return R_C ** 2 / (2 * np.sqrt(np.linalg.det(C_b))) * np.exp(-0.5 * exponent)


def encounter_table(tca, miss, b, pc):
    print(f"{'Случай':>7} {'TCA, с':>12} {'Промах, м':>13} "
          f"{'b₁, м':>12} {'b₂, м':>12} {'P_c':>14} {'P_c ≤ Pдоп':>13}")
    for name, t, distance, point, probability in zip(("A", "B"), tca, miss, b, pc):
        print(f"{name:>7} {t:12.6f} {distance:13.6f} "
              f"{point[0]:12.6f} {point[1]:12.6f} {probability:14.7e} "
              f"{str(probability <= P_DOP):>13}")


def plot_risk_boundary(b, C_b):
    b = np.asarray(b, dtype=float)
    C_b = np.asarray(C_b, dtype=float)
    extent = 1.4 * max(float(np.abs(b).max()),
                       float(np.sqrt(np.linalg.eigvalsh(C_b).max())))
    grid = np.linspace(-extent, extent, 401)
    x, y = np.meshgrid(grid, grid)
    values = pc_approx(np.stack((x, y), axis=-1), C_b)
    fig, ax = plt.subplots(figsize=(6.5, 6.0), constrained_layout=True)
    ax.contour(x, y, values, levels=[P_DOP], colors=INK, linewidths=1.5)
    ax.plot([], [], color=INK, linewidth=1.5, label=r"Граница $P_c=P_{\rm доп}$")
    for name, point, color in zip(("A", "B"), b, (ACCENT, SECOND)):
        ax.scatter(*point, color=color, s=45, zorder=3, label=f"Случай {name}")
        ax.annotate(name, point, xytext=(6, 6), textcoords="offset points", color=color)
    ax.axhline(0, color=INK, linewidth=0.7, alpha=0.35)
    ax.axvline(0, color=INK, linewidth=0.7, alpha=0.35)
    ax.set(xlabel=r"$b_1$, м (ось $X$)", ylabel=r"$b_2$, м (ось $Y$)",
           xlim=(-extent, extent), ylim=(-extent, extent),
           title="Задание 1. Граница допустимой вероятности")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.22)
    ax.legend(loc="best")
    return fig, ax


def maneuver_table(moments, roots, dvt_min, allowed):
    print(f"{'№':>3} {'t_m, с':>13} {'Корень −, м/с':>18} {'Корень +, м/с':>18} "
          f"{'Ближайший, м/с':>19} {'|ΔVₜ| ≤ 0.006':>16}")
    for j, (t, pair, component, valid) in enumerate(zip(moments, roots, dvt_min, allowed), 1):
        print(f"{j:3d} {t:13.6f} {pair[0]:18.9f} {pair[1]:18.9f} "
              f"{component:19.9f} {str(bool(valid)):>16}")


def plot_maneuver_probabilities(b0, C_b, selected_case=None, dv=None):
    b0 = np.asarray(b0, dtype=float)
    C_b = np.asarray(C_b, dtype=float)
    components = np.linspace(-0.04, 0.04, 1001)
    fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
    for j, (g, color) in enumerate(zip(G_T_2, (ACCENT, SECOND)), 1):
        values = pc_approx(b0 + components[:, None] * g, C_b)
        ax.plot(components, values, color=color, linewidth=2,
                label=fr"Момент $t_{{m,{j}}}$")
    ax.axhline(P_DOP, color=INK, linestyle="--", linewidth=1.2,
               label=r"$P_{\rm доп}$")
    ax.axvspan(-DV_MAX, DV_MAX, color=ACCENT, alpha=0.08,
               label=r"Допустимые $\Delta V_t$")
    ax.axvline(-DV_MAX, color=INK, linestyle=":", linewidth=0.9)
    ax.axvline(DV_MAX, color=INK, linestyle=":", linewidth=0.9)
    choice_ready = (_answers_ready((selected_case, ()), (dv, (3,)))
                    and np.asarray(selected_case).dtype.kind in "iu"
                    and int(selected_case) in (1, 2))
    if choice_ready:
        choice = int(selected_case)
        component = float(np.asarray(dv, dtype=float)[1])
        probability = float(pc_approx(b0 + G_T_2[choice - 1] * component, C_b))
        ax.scatter(component, probability, color=INK, s=45, zorder=5,
                    label="Выбранный манёвр")
    elif selected_case is not None or dv is not None:
        print("Для выбранной точки укажите целый номер 1 или 2 и конечный вектор dv_2 формы (3,).")
    ax.set(xlabel=r"Трансверсальная компонента $\Delta V_t$, м/с",
           ylabel=r"Вероятность $P_c$", xlim=(-0.04, 0.04),
           title="Задание 2. Время и величина манёвра")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.grid(alpha=0.22)
    ax.legend(loc="best")
    return fig, ax


In [ ]:
def _report(title, checks):
    print(title)
    for passed, message in checks:
        print(("[OK] " if passed else "[Проверьте] ") + message)
    passed = all(bool(item[0]) for item in checks)
    print("Численные проверки пройдены." if passed
          else "Есть замечания; исправьте отмеченные пункты и повторите проверку.")
    return passed


def check_task1(tca, miss, P, b, C_rho, C_b, pc):
    if any(value is None for value in (tca, miss, P, b, C_rho, C_b, pc)):
        print("Задание 1: сначала заполните tca_1, miss_1, P_1, b_1, C_rho_1, C_b_1 и pc_1.")
        return None
    try:
        tca, miss, P, b, C_rho, C_b, pc = [np.asarray(value, dtype=float)
                                         for value in (tca, miss, P, b, C_rho, C_b, pc)]
    except (TypeError, ValueError):
        print("Нужны числовые массивы без None.")
        return False
    arrays = (tca, miss, P, b, C_rho, C_b, pc)
    shapes = ((2,), (2,), (2, 3), (2, 2), (3, 3), (2, 2), (2,))
    if any(value.shape != shape or not np.all(np.isfinite(value))
           for value, shape in zip(arrays, shapes)):
        print("Проверьте формы массивов из описания и отсутствие NaN или бесконечности.")
        return False
    expected_tca = -(RHO0 @ V_REL) / (V_REL @ V_REL)
    at_tca = RHO0 + expected_tca[:, None] * V_REL
    expected_P = np.array([[1., 0., 0.], [0., 1., 0.]])
    expected_C_rho = C_SC + C_OBJ
    expected_C_b = expected_P @ expected_C_rho @ expected_P.T
    expected_b = at_tca @ expected_P.T
    expected_pc = pc_approx(expected_b, expected_C_b)
    checks = [
        (np.allclose(tca, expected_tca, rtol=0, atol=1e-8),
         "TCA обоих случаев; время измеряется от опорной эпохи t=0."),
        (np.allclose(miss, np.linalg.norm(at_tca, axis=1), rtol=0, atol=1e-7),
         "Минимальные расстояния для A и B."),
        (np.allclose(P, expected_P, rtol=0, atol=1e-10),
         "Проекция на заданные оси X,Y в указанном порядке."),
        (np.allclose(b, expected_b, rtol=0, atol=1e-7),
         "Векторы промаха: строки A,B; положение в TCA, а не в начальную эпоху."),
        (np.allclose(C_rho, expected_C_rho, rtol=1e-8, atol=1e-8),
         "Ковариация относительного положения при независимых ошибках; единицы м²."),
        (np.allclose(C_b, expected_C_b, rtol=1e-8, atol=1e-8),
         "Ковариация в плоскости сближения."),
        (np.allclose(pc, expected_pc, rtol=1e-6, atol=1e-11),
         "Вероятности по заданному приближению, с учётом анизотропии."),
    ]
    return _report("Самопроверка задания 1", checks)


def check_task2(moments, roots, dvt_min, allowed, selected_case, dv, J, b_after, pc_after):
    values = (moments, roots, dvt_min, allowed, selected_case, dv, J, b_after, pc_after)
    if any(value is None for value in values):
        print("Задание 2: сначала заполните моменты, корни, ближайшие компоненты, "
              "допустимость, выбранный случай и результат манёвра.")
        return None
    try:
        moments, roots, dvt_min, dv, b_after = [np.asarray(value, dtype=float)
                                              for value in (moments, roots, dvt_min, dv, b_after)]
        allowed = np.asarray(allowed)
        choice = np.asarray(selected_case)
        J, pc_after = np.asarray(J, dtype=float), np.asarray(pc_after, dtype=float)
    except (TypeError, ValueError):
        print("Проверьте числовые значения и массивы; допустимость задайте булевыми значениями.")
        return False
    arrays = (moments, roots, dvt_min, dv, b_after, J, pc_after)
    shapes = ((2,), (2, 2), (2,), (3,), (2,), (), ())
    if any(value.shape != shape or not np.all(np.isfinite(value))
           for value, shape in zip(arrays, shapes)):
        print("Проверьте формы массивов из описания и отсутствие NaN или бесконечности.")
        return False
    if allowed.shape != (2,) or allowed.dtype.kind != "b":
        print("allowed_2 должен содержать два булевых значения: [True/False, True/False].")
        return False
    if choice.shape != () or choice.dtype.kind not in "iu" or int(choice) not in (1, 2):
        print("selected_case_2 должен быть целым числом 1 или 2.")
        return False
    choice = int(choice)
    tca_B = -(RHO0[1] @ V_REL) / (V_REL @ V_REL)
    b0 = (RHO0[1] + tca_B * V_REL)[:2]
    C_b = (C_SC + C_OBJ)[:2, :2]
    inv = np.linalg.inv(C_b)
    peak = R_C ** 2 / (2 * np.sqrt(np.linalg.det(C_b)))
    level = -2 * np.log(P_DOP / peak)
    expected_roots = []
    for g in G_T_2:
        coefficients = [g @ inv @ g, 2 * b0 @ inv @ g, b0 @ inv @ b0 - level]
        expected_roots.append(np.sort(np.roots(coefficients)))
    expected_roots = np.asarray(expected_roots)
    expected_min = expected_roots[np.arange(2), np.argmin(np.abs(expected_roots), axis=1)]
    expected_allowed = np.abs(expected_min) <= DV_MAX
    valid = np.flatnonzero(expected_allowed)
    expected_choice = int(valid[np.argmin(np.abs(expected_min[valid]))]) + 1
    expected_dv = np.array([0., expected_min[expected_choice - 1], 0.])
    expected_b = b0 + G_T_2[expected_choice - 1] * expected_dv[1]
    checks = [
        (np.allclose(moments, tca_B - LEAD_TIMES_2, rtol=0, atol=1e-7),
         "Оба момента записаны относительно опорной эпохи t=0, а не как время упреждения."),
        (np.allclose(roots, expected_roots, rtol=1e-6, atol=1e-10),
         "Оба корня для каждой эпохи; столбцы — отрицательный и положительный."),
        (np.allclose(dvt_min, expected_min, rtol=1e-6, atol=1e-10),
         "Ближайший к нулю корень сначала выбран без ограничения на величину импульса."),
        (np.array_equal(allowed, expected_allowed),
         "Ограничение |ΔVₜ| ≤ 0.006 м/с проверено для обеих эпох."),
        (choice == expected_choice, "Выбран допустимый момент с наименьшей характеристической скоростью."),
        (np.allclose(dv, expected_dv, rtol=1e-6, atol=1e-10),
         "Полный вектор импульса содержит только трансверсальную компоненту; её знак сохранён."),
        (np.isclose(J, np.linalg.norm(expected_dv), rtol=1e-6, atol=1e-10),
         "J — неотрицательная норма импульса в м/с."),
        (np.allclose(b_after, expected_b, rtol=1e-6, atol=1e-7),
         "Промах после манёвра: коэффициенты G_T_2 уже содержат перенос к сближению."),
        (np.isclose(pc_after, pc_approx(expected_b, C_b), rtol=1e-6, atol=1e-11),
         "Вероятность для выбранного управления по заданной формуле."),
    ]
    return _report("Самопроверка задания 2", checks)


## Задание 1. Какие сближения опаснее?

1. Для случаев A и B найдите момент наибольшего сближения $t_{\mathrm{ca}}$
   при $t\geq0$ и минимальное расстояние. Получите формулу из условия минимума
   квадрата расстояния.
2. Постройте матрицу проекции $P$ на плоскость сближения с осями $X,Y$.
   Найдите $\boldsymbol b_A$ и $\boldsymbol b_B$.
3. Вычислите ковариацию относительного положения $C_\rho$ и её проекцию $C_b$.
   Найдите $P_{c,A}$ и $P_{c,B}$ по заданному приближению.
   Соберите TCA, промахи и вероятности в таблицу; сопоставьте их с допустимым уровнем.
4. Выведите уравнение границы $P_c=P_{\rm доп}$ в плоскости сближения.
   Постройте этот эллипс и отметьте обе точки. Масштабы осей должны быть одинаковыми.
5. Совпадает ли ранжирование двух случаев по геометрическому расстоянию
   и вероятности? Объясните результат с учётом ковариации и направления промаха.


**Момент минимума, матрица проекции и ковариации:**

*Приведите вывод формулы TCA, условие минимума, матрицу $P$
и формулы перехода от ковариаций объектов к $C_\rho$ и $C_b$.*


In [ ]:
tca_1 = None      # массив (2,): TCA для A,B; с
miss_1 = None     # массив (2,): минимальные расстояния для A,B; м
P_1 = None        # массив (2,3): проекция; строки соответствуют X,Y
b_1 = None        # массив (2,2): строки — b_A,b_B; м
C_rho_1 = None    # массив (3,3), м²
C_b_1 = None      # массив (2,2), м²
pc_1 = None       # массив (2,): вероятности A,B по заданной формуле


In [ ]:
if (_answers_ready((tca_1, (2,)), (miss_1, (2,)), (b_1, (2, 2)),
                   (C_rho_1, (3, 3)), (pc_1, (2,)))
        and _covariance_ready(C_b_1)):
    print("C_ρ, м²:\n", np.asarray(C_rho_1))
    print("C_b, м²:\n", np.asarray(C_b_1))
    encounter_table(tca_1, miss_1, b_1, pc_1)
else:
    print("Заполните результаты задания 1 конечными числами в указанных формах; C_b_1 должна быть положительно определённой.")


**Уравнение границы допустимой вероятности:**

*Получите уравнение эллипса из формулы $P_c$. Объясните, какая сторона
границы соответствует допустимой вероятности.*


In [ ]:
if _answers_ready((b_1, (2, 2))) and _covariance_ready(C_b_1):
    fig, ax = plot_risk_boundary(b_1, C_b_1)
    plt.show()
else:
    print("Для графика нужны конечные b_1 формы (2, 2) и положительно определённая C_b_1 формы (2, 2).")


In [ ]:
check_task1(tca_1, miss_1, P_1, b_1, C_rho_1, C_b_1, pc_1)


**Сравнение двух случаев:**

*Сопоставьте расстояния, вероятности и положение точек относительно эллипса.
Объясните, достаточно ли одного геометрического промаха для оценки опасности.*


## Задание 2. Когда и в какую сторону выполнять манёвр?

Используйте случай B: $\boldsymbol b_0=\boldsymbol b_B$, ковариация $C_b$
сохраняется. Моменты манёвра отсчитываются от той же опорной эпохи $t=0$
и могут предшествовать ей:
$$t_{m,1}=t_{\mathrm{ca},B}-600\ \text{с},\qquad
  t_{m,2}=t_{\mathrm{ca},B}-1800\ \text{с}.$$
Разрешён только трансверсальный импульс
$\Delta\boldsymbol V=(0,\Delta V_t,0)^\top$,
$|\Delta V_t|\leq0{,}006$ м/с. Его влияние уже задано:
$$\boldsymbol b(\Delta V_t;t_{m,1})=\boldsymbol b_0+
  \begin{pmatrix}1200\\600\end{pmatrix}\Delta V_t,\qquad
  \boldsymbol b(\Delta V_t;t_{m,2})=\boldsymbol b_0+
  \begin{pmatrix}3600\\-1200\end{pmatrix}\Delta V_t.$$
Коэффициенты имеют единицы секунд и включают полный перенос импульса
к сближению; повторно умножать их на 600 или 1800 секунд нельзя.

1. Для каждого момента выведите уравнение $P_c(\Delta V_t)=P_{\rm доп}$
   и найдите **оба корня**. Сначала не учитывайте ограничение на величину импульса.
2. В каждой паре выберите ближайший к нулю корень и затем проверьте
   $|\Delta V_t|\leq0{,}006$ м/с. Сведите моменты, корни и допустимость в таблицу.
3. Выберите допустимый момент с наименьшим $J=\|\Delta\boldsymbol V\|$.
   Укажите полный вектор импульса, $J$, новый вектор промаха и вероятность.
4. На одном графике постройте обе зависимости $P_c(\Delta V_t)$
   на $[-0{,}04;0{,}04]$ м/с. Отметьте порог, допустимую область импульсов
   и выбранную точку.
5. Объясните роль времени упреждения и знака импульса. Почему уменьшение
   вероятности нужно проверять с учётом направления смещения, а не только его модуля?


In [ ]:
# Используются только ваши результаты задания 1. Неполные данные сбрасывают связь.
b0_2, C_b_2 = None, None
if _answers_ready((b_1, (2, 2))) and _covariance_ready(C_b_1):
    b0_2 = np.asarray(b_1, dtype=float)[1].copy()
    C_b_2 = np.asarray(C_b_1, dtype=float).copy()
else:
    print("Для задания 2 сначала заполните b_1 и корректную C_b_1 в задании 1.")


**Уравнения для границы вероятности и выбор корня:**

*Для обеих эпох запишите зависимость квадратичной формы от $\Delta V_t$,
получите уравнения и найдите корни. Объясните, как выбирается корень
минимального модуля и в какой момент применяется ограничение.*


In [ ]:
moments_2 = None        # массив (2,): t_m1,t_m2 относительно опорной эпохи t=0; с
roots_2 = None          # массив (2,2): строки — эпохи, столбцы — отрицательный/положительный корни; м/с
dvt_min_2 = None        # массив (2,): ближайшие к нулю корни ДО ограничения; м/с
allowed_2 = None        # два булевых значения: удовлетворяют ли эти корни ограничению


In [ ]:
if _answers_ready((moments_2, (2,)), (roots_2, (2, 2)),
                  (dvt_min_2, (2,)), (allowed_2, (2,))):
    maneuver_table(moments_2, roots_2, dvt_min_2, allowed_2)
else:
    print("Заполните моменты, оба корня, ближайшие компоненты и допустимость без None или нечисловых значений.")


**Выбор допустимого манёвра:**

*Обоснуйте выбор момента. Запишите вектор импульса со знаком,
характеристическую скорость и результаты проверки промаха и вероятности.*


In [ ]:
selected_case_2 = None  # целое число 1 или 2
dv_2 = None             # массив (3,): ΔV_r,ΔV_t,ΔV_n; м/с
J_2 = None              # норма импульса; м/с
b_after_2 = None        # массив (2,): новый промах в осях X,Y; м
pc_after_2 = None       # вероятность для выбранного управления


In [ ]:
if (_answers_ready((selected_case_2, ()), (dv_2, (3,)), (J_2, ()),
                   (b_after_2, (2,)), (pc_after_2, ()))
        and np.asarray(selected_case_2).dtype.kind in "iu"
        and int(selected_case_2) in (1, 2)):
    print(f"Выбран момент t_m,{int(selected_case_2)}.")
    print("ΔV, м/с:", np.asarray(dv_2))
    print(f"J = {float(J_2):.9f} м/с")
    print("b после манёвра, м:", np.asarray(b_after_2))
    print(f"P_c = {float(pc_after_2):.9e}")
else:
    print("Заполните выбранный номер 1 или 2, конечный импульс формы (3,) и результаты манёвра.")


In [ ]:
if _answers_ready((b0_2, (2,))) and _covariance_ready(C_b_2):
    fig, ax = plot_maneuver_probabilities(b0_2, C_b_2, selected_case_2, dv_2)
    plt.show()
else:
    print("Для графика заполните b_1 и C_b_1 в задании 1, затем повторите ячейку b0_2/C_b_2.")


In [ ]:
check_task2(moments_2, roots_2, dvt_min_2, allowed_2,
            selected_case_2, dv_2, J_2, b_after_2, pc_after_2)


**Влияние времени и направления манёвра:**

*Объясните различие результатов двух эпох. Свяжите знак импульса,
положение точки в плоскости сближения и границу допустимой вероятности.*


Чекеры проверяют только численные результаты для заданных исходных данных.
Графики иллюстрируют зависимости на сетке. Эти проверки
**не заменяют выводов и объяснений**.

## Что представить

Сдайте этот ноутбук с полными решениями двух заданий:
выводом формулы TCA, матрицей проекции и ковариациями, таблицей сравнения
сближений, уравнением и графиком границы вероятности, расчётом обоих корней
для каждой эпохи, проверкой ограничений и выбором манёвра.
Добавьте график двух вероятностей с порогом, допустимой областью
и выбранной точкой, а также пояснения к обоим заданиям.
Использование чекеров и сохранение результатов их вызова не требуются.

Перед отправкой перезапустите ядро, выполните ячейки решения по порядку и сохраните
ноутбук с результатами. Название файла: **`hw_6_name_surname.ipynb`**;
вместо `name_surname` укажите свои имя и фамилию латиницей.
